In [1]:
import numpy as np


# CONSTANTS
FS = 1_000_000          # Sampling Frequency (1 MHz)
C = 3e8                 # Speed of light


# LOAD DATASETS

data_A = np.load("dataset_A_capture.npy")
data_B = np.load("dataset_B_capture.npy")


# ADAPTIVE CFAR DETECTION
def adaptive_cfar(signal_mag,
                  guard=12,
                  train=80,
                  scale=2.4):

    power = signal_mag ** 2

    threshold = np.zeros_like(power)

    for i in range(train + guard,
                   len(power) - train - guard):

        left = power[i - train - guard : i - guard]

        right = power[i + guard : i + guard + train]

        noise_cells = np.concatenate((left, right))

        noise_estimate = np.median(noise_cells)

        threshold[i] = scale * noise_estimate

    threshold = np.sqrt(threshold)

    valid = threshold[train + guard : -(train + guard)]

    edge_val = np.median(valid)

    threshold[: train + guard] = edge_val

    threshold[-(train + guard):] = edge_val

    detections = signal_mag > threshold

    return detections, threshold


# CLEAN DETECTION MASK

def clean_detection_mask(mask):

    clean = mask.copy().astype(int)

    for i in range(2, len(clean) - 2):

        if np.sum(clean[i-2 : i+3]) < 2:
            clean[i] = 0

    return clean.astype(bool)


# EXTRACT PULSES

def extract_pulses(signal_mag,
                   mask,
                   threshold):

    min_width = 3
    max_width = 400

    pulse_locations = []

    i = 0

    while i < len(mask):

        if mask[i]:

            start = i

            while i < len(mask) and mask[i]:
                i += 1

            end = i

            width = end - start

            if width < min_width or width > max_width:
                continue

            energy = np.sum(signal_mag[start:end])

            local_threshold = 3 * np.mean(threshold[start:end])

            if energy < local_threshold:
                continue

            peak_ratio = (
                np.max(signal_mag[start:end]) /
                (np.mean(signal_mag[start:end]) + 1e-9)
            )

            if peak_ratio < 1.5:
                continue

            peak_idx = start + np.argmax(signal_mag[start:end])

            pulse_locations.append(peak_idx)

        else:
            i += 1

    return np.array(pulse_locations)


# PRI ESTIMATION

def estimate_pris(pulses):

    if len(pulses) < 5:
        return np.array([]), {}

    pri_list = []

    for order in range(1, 6):

        diffs = pulses[order:] - pulses[:-order]

        pri_ms = (diffs / FS) * 1000

        pri_list.extend(pri_ms)

    pri_list = np.array(pri_list)

    pri_list = pri_list[
        (pri_list > 0.05) &
        (pri_list < 20)
    ]

    if len(pri_list) == 0:
        return np.array([]), {}

    bins = np.arange(0.05, 20.01, 0.01)

    hist, edges = np.histogram(pri_list, bins=bins)

    vote_threshold = 0.15 * np.max(hist)

    peak_bins = np.where(hist > vote_threshold)[0]

    candidate_pris = []

    pri_strength = {}

    for p in peak_bins:

        pri = (edges[p] + edges[p+1]) / 2

        candidate_pris.append(pri)

        pri_strength[pri] = hist[p]

    return np.array(candidate_pris), pri_strength


# PRI CLUSTERING + HARMONIC SUPPRESSION

def cluster_pris(pris,
                 pri_strength,
                 tolerance=0.12):

    if len(pris) == 0:
        return np.array([])

    pris = np.sort(pris)

    clusters = []

    current = [pris[0]]

    for p in pris[1:]:

        if abs(p - np.mean(current)) < tolerance:
            current.append(p)

        else:
            clusters.append(np.mean(current))
            current = [p]

    clusters.append(np.mean(current))

    filtered = []

    base_pri = np.min(pris)

    for c in clusters:

        # Harmonic rejection
        if c > 5 * base_pri:
            continue

        harmonic = False

        for f in filtered:

            ratio = c / f

            if abs(ratio - round(ratio)) < 0.08:

                c_strength = pri_strength.get(
                    min(pri_strength.keys(),
                        key=lambda x: abs(x-c)),
                    1
                )

                f_strength = pri_strength.get(
                    min(pri_strength.keys(),
                        key=lambda x: abs(x-f)),
                    1
                )

                if c_strength < 0.6 * f_strength:
                    harmonic = True
                    break

        if not harmonic:
            filtered.append(c)

    return np.round(filtered, 3)


# GROUP PULSES BY PRI

def group_pulses_by_pri(
        pulse_starts,
        pulse_intervals_ms,
        detected_pris,
        tolerance=0.2):

    emitters = {pri: [] for pri in detected_pris}

    for i in range(len(pulse_intervals_ms)):

        interval = pulse_intervals_ms[i]

        nearest = detected_pris[
            np.argmin(
                np.abs(detected_pris - interval)
            )
        ]

        if abs(nearest - interval) > tolerance:
            continue

        emitters[nearest].append(
            pulse_starts[i]
        )

    for pri in emitters:

        emitters[pri] = sorted(
            list(set(emitters[pri]))
        )

    return emitters


# CARRIER FREQUENCY ESTIMATION

def estimate_carrier_frequency(signal_segment):

    N = len(signal_segment)

    window = np.blackman(N)

    nfft = 16384

    spectrum = np.fft.fftshift(
        np.fft.fft(
            signal_segment.astype(complex) * window,
            n=nfft
        )
    )

    freqs = np.fft.fftshift(
        np.fft.fftfreq(nfft, d=1/FS)
    )

    magnitude = np.abs(spectrum)

    center = nfft // 2

    magnitude[: center + 50] = 0

    peak_idx = np.argmax(magnitude)

    if 0 < peak_idx < len(magnitude) - 1:

        alpha = magnitude[peak_idx - 1]
        beta = magnitude[peak_idx]
        gamma = magnitude[peak_idx + 1]

        denom = alpha - 2*beta + gamma

        correction = (
            0.5 * (alpha - gamma) / denom
            if denom != 0 else 0.0
        )

    else:
        correction = 0.0

    freq_resolution = freqs[1] - freqs[0]

    return (
        freqs[peak_idx]
        + correction * freq_resolution
    )


# DoA ESTIMATION

def estimate_doa(data,
                 pulse_idx,
                 carrier_freq):

    if abs(carrier_freq) < 5e4:
        return np.nan

    wavelength = C / abs(carrier_freq)

    d = wavelength / 2

    start = max(0, pulse_idx - 64)

    end = min(data.shape[1], pulse_idx + 64)

    peak_offset = np.argmax(
        np.abs(data[0, start:end])
    )

    snapshot = np.mean(
        data[
            :,
            start + peak_offset - 8 :
            start + peak_offset + 9
        ],
        axis=1
    )

    phase_diffs = np.angle(
        snapshot[1:] * np.conj(snapshot[:-1])
    )

    phase_diff = np.mean(phase_diffs)

    argument = np.clip(
        (phase_diff * wavelength)
        / (2 * np.pi * d),
        -1,
        1
    )

    doa = np.degrees(np.arcsin(argument))

    if np.isnan(doa) or abs(doa) > 85:
        return np.nan

    return doa


# MAIN PROCESSING

def process_dataset(data, label):

    signal = data[0, :]

    mag = np.abs(signal)

    # CFAR DETECTION

    mask, threshold = adaptive_cfar(mag)

    mask = clean_detection_mask(mask)

    false_alarm_rate = np.mean(
        mask & (mag < threshold * 1.1)
    )

    # PULSE EXTRACTION

    pulse_starts = extract_pulses(
        mag,
        mask,
        threshold
    )

    if len(pulse_starts) < 5:

        print("\n" + "=" * 70)
        print(f"DATASET : {label}")
        print("=" * 70)

        print("\nToo few pulses detected.")

        return []

    # PRI ESTIMATION

    pulse_intervals_ms = (
        np.diff(pulse_starts) / FS
    ) * 1000

    pulse_intervals_ms = pulse_intervals_ms[
        pulse_intervals_ms > 0.05
    ]

    raw_pris, pri_strength = estimate_pris(
        pulse_starts
    )

    detected_pris = cluster_pris(
        raw_pris,
        pri_strength
    )

    if len(detected_pris) == 0:

        print("\nNo valid PRI values found.")

        return []

    # PRI GROUPING

    emitters = group_pulses_by_pri(
        pulse_starts,
        pulse_intervals_ms,
        detected_pris
    )

    # EMITTER CHARACTERIZATION

    results = []

    window = 4096

    for pri, pulse_list in emitters.items():

        if len(pulse_list) < 5:
            continue

        pulse_freqs = []

        pulse_doas = []

        for ps in pulse_list[:10]:

            start = max(0, ps - 256)

            end = min(len(signal), ps + window)

            seg = signal[start:end]

            fc = estimate_carrier_frequency(seg)

            pulse_freqs.append(fc)

            doa = estimate_doa(data, ps, fc)

            pulse_doas.append(doa)

        mean_fc = np.mean(pulse_freqs)

        freq_std = np.std(pulse_freqs)

        if freq_std > 5e5:
            continue

        valid_doas = [
            d for d in pulse_doas
            if not np.isnan(d)
        ]

        mean_doa = (
            np.mean(valid_doas)
            if valid_doas else np.nan
        )

        doa_std = (
            np.std(valid_doas)
            if len(valid_doas) > 1 else np.nan
        )

        pri_local = []

        for i in range(len(pulse_list)-1):

            interval = (
                (pulse_list[i+1] - pulse_list[i])
                / FS
            ) * 1000

            # Normalize missed-pulse multiples
            interval = interval / max(
                1,
                round(interval / pri)
            )

            pri_local.append(interval)

        pri_std = (
            np.std(pri_local)
            if len(pri_local) > 1 else np.nan
        )

        confidence = (
          min(1.0, len(pulse_list)/15)
          * np.exp(-freq_std/2.5e5)
      )

        results.append({

            "PRI_ms": pri,
            "Carrier_Frequency_Hz": mean_fc,
            "DOA_deg": mean_doa,
            "Pulse_Count": len(pulse_list),

            "PRI_STD_ms": pri_std,
            "FC_STD_Hz": freq_std,
            "DOA_STD_deg": doa_std,

            "Confidence": confidence
        })

    # DEDUPLICATION

    deduped = []

    for r in results:

        matched = False

        for ex in deduped:

            fc_match = (
                abs(
                    r["Carrier_Frequency_Hz"]
                    - ex["Carrier_Frequency_Hz"]
                ) < 50_000
            )

            doa_match = (
                abs(
                    r["DOA_deg"]
                    - ex["DOA_deg"]
                ) < 10
                if not np.isnan(r["DOA_deg"])
                and not np.isnan(ex["DOA_deg"])
                else False
            )

            if fc_match and doa_match:

                matched = True
                break

        if not matched:
            deduped.append(dict(r))

    final_results = [

        r for r in deduped

        if r["Pulse_Count"] >= 5
        and abs(r["Carrier_Frequency_Hz"]) > 5e3
    ]

    # FINAL STRUCTURED REPORT

    print("\n")
    print("=" * 70)
    print("RADAR EMITTER ANALYSIS REPORT")
    print("=" * 70)

    print(f"\nDataset Processed : {label}")
    print(f"Detected Pulses   : {len(pulse_starts)}")
    print(f"False Alarm Rate  : {false_alarm_rate:.4f}")

    print("\nDetected PRI Candidates (ms):")

    if len(detected_pris) > 0:

        for p in detected_pris:
            print(f"  • {p:.3f} ms")

    else:
        print("  None")

    print("\n" + "-" * 70)

    # NO EMITTERS FOUND

    if len(final_results) == 0:

        print("\nNo high-confidence emitters reconstructed.")

        print("\nPossible Causes:")
        print("  • Severe interleaving")
        print("  • Low Signal-to-Noise Ratio")
        print("  • Multipath interference")
        print("  • Insufficient pulse grouping")

        print("\n" + "=" * 70)

        return final_results

    # EMITTER REPORT

    for idx, r in enumerate(final_results):

        if r["Confidence"] > 0.7:
            status = "HIGH CONFIDENCE"

        elif r["Confidence"] > 0.3:
            status = "MODERATE CONFIDENCE"

        else:
            status = "LOW CONFIDENCE"

        print(f"\nEmitter {idx + 1}")

        print("-" * 70)

        print(f"{'Carrier Frequency':<35}"
              f"{r['Carrier_Frequency_Hz']:.2f} Hz")

        print(f"{'Pulse Repetition Interval':<35}"
              f"{r['PRI_ms']:.3f} ms")

        print(f"{'Direction of Arrival':<35}"
              f"{r['DOA_deg']:.2f} deg")

        print(f"{'Detected Pulses':<35}"
              f"{r['Pulse_Count']}")

        print(f"{'PRI Stability (STD)':<35}"
              f"{r['PRI_STD_ms']:.4f} ms")

        print(f"{'Frequency Stability (STD)':<35}"
              f"{r['FC_STD_Hz']:.2f} Hz")

        print(f"{'DoA Stability (STD)':<35}"
              f"{r['DOA_STD_deg']:.2f} deg")

        print(f"{'Confidence Score':<35}"
              f"{r['Confidence']:.3f}")

        print(f"{'Detection Reliability':<35}"
              f"{status}")

        # INTERPRETATION

        print("\nInterpretation:")

        if r["PRI_STD_ms"] < 0.05:
            print("  • Stable PRI stream reconstruction")

        else:
            print("  • PRI stream contains timing variation")

        if r["DOA_STD_deg"] < 10:
            print("  • Spatial estimation is stable")

        else:
            print("  • DoA estimate affected by phase noise")

        if r["FC_STD_Hz"] < 150000:
            print("  • Carrier frequency estimation coherent")

        else:
            print("  • Frequency estimate shows spectral spread")

        print("-" * 70)

    # DATASET SUMMARY

    print("\nDATASET SUMMARY")

    print("-" * 70)

    print(f"{'Total Emitters Found':<35}"
          f"{len(final_results)}")

    print(f"{'Total Pulses Detected':<35}"
          f"{len(pulse_starts)}")

    print(f"{'Estimated False Alarm Rate':<35}"
          f"{false_alarm_rate:.4f}")

    print("=" * 70)

    return final_results

# RUN ANALYSIS

results_A = process_dataset(data_A, "Dataset A")

results_B = process_dataset(data_B, "Dataset B")



RADAR EMITTER ANALYSIS REPORT

Dataset Processed : Dataset A
Detected Pulses   : 38
False Alarm Rate  : 0.0278

Detected PRI Candidates (ms):
  • 0.985 ms

----------------------------------------------------------------------

Emitter 1
----------------------------------------------------------------------
Carrier Frequency                  286371.27 Hz
Pulse Repetition Interval          0.985 ms
Direction of Arrival               -2.89 deg
Detected Pulses                    9
PRI Stability (STD)                0.0080 ms
Frequency Stability (STD)          111575.81 Hz
DoA Stability (STD)                19.36 deg
Confidence Score                   0.384
Detection Reliability              MODERATE CONFIDENCE

Interpretation:
  • Stable PRI stream reconstruction
  • DoA estimate affected by phase noise
  • Carrier frequency estimation coherent
----------------------------------------------------------------------

DATASET SUMMARY
--------------------------------------------------------